In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "BNBUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,imbalance,imbalance_5,imbalance_15,trend_strength,vol_regime_ratio,is_trending,is_high_vol,mom_x_imb,mr_x_vol,trend_x_imb
0,2025-09-01 00:00:00+00:00,857.66,857.67,857.24,857.66,251.305,2025-09-01 00:00:59.999999+00:00,215467.75012,654,192.217,...,0.529751,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
1,2025-09-01 00:01:00+00:00,857.67,858.16,857.67,858.15,140.110,2025-09-01 00:01:59.999999+00:00,120206.45623,490,82.628,...,0.179473,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
2,2025-09-01 00:02:00+00:00,858.16,858.16,857.55,857.75,207.449,2025-09-01 00:02:59.999999+00:00,177947.04945,566,66.245,...,-0.361337,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
3,2025-09-01 00:03:00+00:00,857.76,858.25,857.75,857.81,315.626,2025-09-01 00:03:59.999999+00:00,270770.38427,391,254.225,...,0.610926,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
4,2025-09-01 00:04:00+00:00,857.80,857.81,856.12,856.13,415.090,2025-09-01 00:04:59.999999+00:00,355712.34813,1816,55.089,...,-0.734568,0.044849,NaN,NaN,NaN,0,0,NaN,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-18 12:40:01,605] A new study created in memory with name: no-name-f6530026-432e-4310-ae12-88032421260a


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:07<?, ?it/s]

Best trial: 0. Best value: 0.00228908:   0%|          | 0/50 [00:07<?, ?it/s]

Best trial: 0. Best value: 0.00228908:   2%|▏         | 1/50 [00:07<06:00,  7.36s/it]

...

Best trial: 36. Best value: 0.0233227:  98%|█████████▊| 49/50 [12:32<00:37, 37.72s/it]

Best trial: 36. Best value: 0.0233227:  98%|█████████▊| 49/50 [12:32<00:37, 37.72s/it]

Best trial: 36. Best value: 0.0233227: 100%|██████████| 50/50 [12:32<00:00, 34.93s/it]

Best trial: 36. Best value: 0.0233227: 100%|██████████| 50/50 [12:32<00:00, 15.04s/it]

[I 2026-03-18 12:52:33,669] Trial 49 finished with value: 0.008912077263673194 and parameters: {'n_estimators': 500, 'max_depth': 9, 'min_samples_split': 19, 'min_samples_leaf': 19, 'max_features': 1.0, 'bootstrap': False}. Best is trial 36 with value: 0.02332272996418802.

[optuna] best trial
value: 0.023323
params:
  n_estimators: 200
  max_depth: 11
  min_samples_split: 2
  min_samples_leaf: 16
  max_features: 1.0
  bootstrap: False


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final rf...


[training] done in 20.71s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.278432
Test IC:       0.005981
Train Rank IC: 0.024365
Test Rank IC:  -0.015541
Train RMSE:    0.002091
Test RMSE:     0.001684


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
vol_15              0.329243
mom_3               0.326133
range_15            0.057646
vol_regime_ratio    0.050937
vol_30              0.049455
imbalance_5         0.038300
mom_10              0.038016
mom_15              0.030756
trend_strength      0.020920
dist_ma_30          0.013724
bar_range           0.013059
trades_z            0.008164
vol_ratio_5_30      0.006211
mom_5               0.005891
vol_5               0.005230
mom_x_imb           0.004113
volume_mom_5        0.002203
num_trades_mom_5    0.000000
dist_ma_15          0.000000
dist_ma_5           0.000000
dist_ma_15_z        0.000000
range_ratio         0.000000
is_high_vol         0.000000
is_trending         0.000000
range_5             0.000000
imbalance           0.000000
volume_z            0.000000
imbalance_15        0.000000
mr_x_vol            0.000000
trend_x_imb         0.000000
dtype: float64


In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/rf/BNBUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/rf/BNBUSDT__h5_model.joblib
[saved] features -> models/rf/BNBUSDT__h5_feature_cols.json
[saved] feature importance -> models/rf/BNBUSDT__h5_feature_importance.csv
[saved] metadata -> models/rf/BNBUSDT__h5_meta.json
